# Customer Churn / E-commerce Sales Trend Analysis
**Level:** Intermediate | **Estimated time:** 8 hours  
**Dataset:** Generated synthetic e-commerce order data (not real customer information).

## Business objective
Analyse sales performance, identify monthly revenue trends, strongest product categories, and high-value customers. Use the findings to recommend practical business actions.

## Questions
1. How much revenue was generated?
2. Which months and categories performed best?
3. Who are the most valuable customers?
4. How concentrated is revenue among top customers?
5. What should the business do next?

> **Important:** This project uses generated data for learning. The results are illustrative and must not be treated as real market findings.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load the dataset

In [ ]:
# Keep ecommerce_sales_generated.csv in the same folder as this notebook.
df = pd.read_csv("ecommerce_sales_generated.csv")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Data quality checks and cleaning

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df = df.dropna(subset=["Order_ID", "Customer_ID", "Order_Date", "Quantity", "Unit_Price"])
df["Category"] = df["Category"].fillna("Unknown")
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")
df = df.dropna(subset=["Quantity", "Unit_Price"])
df["Revenue"] = df["Quantity"] * df["Unit_Price"]

print("Shape after cleaning:", df.shape)
print("Missing values after cleaning:")
display(df.isna().sum())

## 4. Feature engineering: extract date features

In [ ]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Year_Month"] = df["Order_Date"].dt.to_period("M").astype(str)

display(df[["Order_Date", "Year", "Month", "Month_Name", "Year_Month"]].head())

## 5. Overall business KPIs

In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = df["Order_ID"].nunique()
total_customers = df["Customer_ID"].nunique()
average_order_value = df.groupby("Order_ID")["Revenue"].sum().mean()

kpis = pd.DataFrame({
    "Metric": ["Total revenue", "Unique orders", "Unique customers", "Average order value"],
    "Value": [total_revenue, total_orders, total_customers, average_order_value]
})
display(kpis)

## 6. Monthly revenue trend and pivot table

In [ ]:
monthly_sales = (df.pivot_table(index="Year_Month", values="Revenue", aggfunc="sum")
                   .sort_index())
display(monthly_sales.tail(12))

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales["Revenue"], marker="o")
plt.title("Monthly revenue trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 7. Category performance

In [ ]:
category_sales = (df.pivot_table(index="Category", values="Revenue", aggfunc="sum")
                    .sort_values("Revenue", ascending=False))
display(category_sales)

plt.figure(figsize=(9, 5))
sns.barplot(data=category_sales.reset_index(), x="Category", y="Revenue")
plt.title("Revenue by product category")
plt.xlabel("Product category")
plt.ylabel("Revenue")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Top customers and Pareto analysis

In [ ]:
customer_sales = (df.groupby("Customer_ID")["Revenue"].sum()
                    .sort_values(ascending=False)
                    .reset_index())
customer_sales["Cumulative_Revenue"] = customer_sales["Revenue"].cumsum()
customer_sales["Cumulative_Revenue_Pct"] = (
    customer_sales["Cumulative_Revenue"] / customer_sales["Revenue"].sum() * 100
)
display(customer_sales.head(10))

top_20_count = max(1, int(np.ceil(len(customer_sales) * 0.20)))
top_20_share = customer_sales.head(top_20_count)["Revenue"].sum() / customer_sales["Revenue"].sum() * 100
print(f"Top 20% of customers account for {top_20_share:.1f}% of revenue.")

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(customer_sales)+1), customer_sales["Cumulative_Revenue_Pct"])
plt.axhline(80, linestyle="--", label="80% revenue reference")
plt.xlabel("Customers ranked by revenue")
plt.ylabel("Cumulative revenue (%)")
plt.title("Customer revenue concentration (Pareto analysis)")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Year-over-year revenue comparison

In [ ]:
yearly_sales = df.groupby("Year")["Revenue"].sum()
display(yearly_sales.to_frame("Revenue"))

if 2024 in yearly_sales.index and 2025 in yearly_sales.index:
    growth = (yearly_sales.loc[2025] / yearly_sales.loc[2024] - 1) * 100
    print(f"Revenue change from 2024 to 2025: {growth:.1f}%")

## 10. Business recommendations

In [ ]:
# Use the calculated results above to write evidence-based recommendations.
# Example prompts:
# - Which category deserves more promotion or inventory?
# - Which months need campaign planning?
# - Could a loyalty programme help retain high-value customers?
# - How can the business reduce over-dependence on a small customer group?

## 11. Conclusion
This analysis demonstrates how order-level data can be transformed into business insights using Python, Pandas, pivot tables, date features, visualizations, and customer revenue ranking.

**Limitations:** The dataset is synthetic. It does not include marketing spend, profit margin, returns, discounts, customer demographics, or churn labels. Therefore, this project analyses sales trends and customer value; it does **not** calculate actual customer churn.